In [1]:
import os
import cv2
import numpy as np
from PIL import Image

In [24]:
def remove_shadows_and_noise(image):
    """
    Удаляет тени и шум, мягко снижает контраст для лучшей читаемости текста.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Удаление теней — делим изображение на фон
    dilated = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    no_shadows = cv2.divide(gray, dilated, scale=255)

    # Сглаживаем шумы
    blurred = cv2.GaussianBlur(no_shadows, (5, 5), 0)

    # 🔽 Понижение контрастности вручную (alpha < 1)
    alpha = 1  # коэффициент контрастности (1.0 — без изменений)
    beta = 10    # добавим немного яркости

    # adjusted = cv2.convertScaleAbs(blurred, alpha=alpha)
    result = cv2.convertScaleAbs(blurred, alpha=alpha, beta=beta)

    # Перевод в 3 канала, если нужно сохранить цвет
    # result = cv2.cvtColor(adjusted, cv2.COLOR_GRAY2BGR)
    return result

In [42]:
def correct_rotation_to_vertical(image):
    """
    Делает изображение вертикальным: если оно горизонтальное — поворачивает на 90°.
    """
    h, w = image.shape[:2]
    if w > h:
        image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    return image

def process_jpg_images(input_folder, output_folder):
    """
    Обрабатывает .jpg: чистка + поворот в вертикальную ориентацию.
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".jpg"):
            input_path = os.path.join(input_folder, filename)
            image = cv2.imread(input_path)

            if image is None:
                print(f"❌ Не удалось открыть: {filename}")
                continue

            cleaned = remove_shadows_and_noise(image)
            tmp = smooth_contrast(cleaned)
            tmp = apply_clahe(tmp)
            rotated = correct_rotation_to_vertical(tmp)

            # preprocessed = preprocess_document(image)

            output_path = os.path.join(output_folder, filename)
            cv2.imwrite(output_path, rotated)
            print(f"✅ Обработан: {filename}")

In [43]:
input_dir = "../data/PRIVATE/jpg"
output_dir = "../data/PRIVATE/output_jpg"
process_jpg_images(input_dir, output_dir)

✅ Обработан: a_2_1.jpg
✅ Обработан: a4-3.jpg
✅ Обработан: a6-5.jpg
✅ Обработан: a_1_3.jpg
✅ Обработан: a4-5.jpg
✅ Обработан: a6-3.jpg
✅ Обработан: a_1_1.jpg
✅ Обработан: a5-5.jpg
✅ Обработан: a5-1.jpg
✅ Обработан: a3-2.jpg
✅ Обработан: a5-4.jpg
✅ Обработан: a_2_3.jpg
✅ Обработан: a_2_2.jpg
✅ Обработан: a4-1.jpg
✅ Обработан: a6-2.jpg
✅ Обработан: a4-2.jpg
✅ Обработан: a5-3.jpg
✅ Обработан: a5-2.jpg
✅ Обработан: a4-4.jpg
✅ Обработан: a6-1.jpg
✅ Обработан: a3-4.jpg
✅ Обработан: a_2_5.jpg
✅ Обработан: a3-1.jpg
✅ Обработан: a3-5.jpg
✅ Обработан: a_1_5.jpg
✅ Обработан: a6-4.jpg
✅ Обработан: a_1_4.jpg
✅ Обработан: a_2_4.jpg
✅ Обработан: a_1_2.jpg
✅ Обработан: a3-3.jpg


#################################################################################################

In [2]:
def ensure_vertical(image):
    h, w = image.shape[:2]
    if w > h:
        image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    return image

In [3]:
def order_points(pts):
    """
    Упорядочивает 4 точки в порядке: [верх-лево, верх-право, низ-право, низ-лево]
    """
    rect = np.zeros((4, 2), dtype="float32")

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # верхний левый угол
    rect[2] = pts[np.argmax(s)]  # нижний правый угол

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # верхний правый угол
    rect[3] = pts[np.argmax(diff)]  # нижний левый угол

    return rect

In [4]:
def four_point_transform(image, rect):
    """
    Делает перспективное преобразование изображения (вид сверху на документ).
    """
    (tl, tr, br, bl) = rect

    # Вычисляем ширину и высоту нового изображения
    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = int(max(heightA, heightB))

    # Точки назначения (прямоугольник)
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype="float32")

    # Матрица преобразования и warp
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))

    return warped

In [12]:
def crop_to_document(image):
    # gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(image, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)
        if len(approx) == 4:
            doc_cnt = approx
            break
    else:
        return image  # контур не найден

    pts = doc_cnt.reshape(4, 2)
    rect = order_points(pts)
    dst = four_point_transform(image, rect)
    return dst

In [6]:
def remove_shadows(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    dilated = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    normalized = cv2.divide(gray, dilated, scale=255)
    return normalized

In [26]:
def smooth_contrast(image_gray):
    blurred = cv2.GaussianBlur(image_gray, (3, 3), 0)
    adjusted = cv2.convertScaleAbs(blurred, alpha=0.9, beta=10)
    return adjusted

In [30]:
def apply_clahe(image_gray):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(image_gray)

In [9]:
def preprocess_document(image):
    image = remove_shadows(image)
    image = crop_to_document(image)
    image = ensure_vertical(image)
    # image = remove_shadows(image)
    image = smooth_contrast(image)
    image = apply_clahe(image)
    # image = prepare_for_ocr(image)
    return image